# Algorithm design

Fig. 2(e) and Fig. 2(f). Match consumers under population-feedback distribution to consumers under individual-feedback distribution.

### 1. The two arms

Focal is the treated arm. Comparison is the other arm. Nothing is paired yet.

In [1]:
from pathlib import Path
import sys

here = Path.cwd().resolve()
for candidate in [here, *here.parents]:
    if (candidate / "run_pipeline.py").exists():
        sys.path.insert(0, str(candidate))
        root = candidate
        break
else:
    raise FileNotFoundError("Open this notebook from the repository.")

from core.pipeline import ensure_synthetic, load_config

cfg = load_config(root)
spec = cfg["tasks"]["algorithm"]
pool = ensure_synthetic(root, cfg, rebuild=False)[{"creator": "creators", "consumer": "consumers", "video": "videos", "algorithm": "algorithm"}["algorithm"]]
focal = pool[pool["treated"] == 1].reset_index(drop=True)
comparison = pool[pool["treated"] == 0].reset_index(drop=True)
import pandas as pd
pd.DataFrame([{
    "focal_label": spec["focal"],
    "comparison_label": spec["comparison"],
    "n_focal": len(focal),
    "n_comparison": len(comparison),
    "exact": ", ".join(spec["exact"]),
    "caliper": ", ".join(f"{name} <= {limit}" for name, limit in (spec.get("caliper") or {}).items()) or "none",
}])


,focal_label,comparison_label,n_focal,n_comparison,exact,caliper
0,population-feedback,individual-feedback,1200,1200,"city, gender",none


### 2. Exact cells

Units are blocked on the categorical covariates first. A focal unit whose cell has no comparison unit cannot be matched, before any distance is computed.

In [2]:
from core.matching import exact_cell_counts

cells = exact_cell_counts(focal, comparison, spec["exact"])
cells.sort_values(["n_comparison", "n_focal"], ascending=[True, False])


,city,gender,n_focal,n_comparison
1,east,male,196,173
4,south,female,185,191
5,south,male,194,199
2,north,female,191,209
3,north,male,228,211
0,east,female,206,217


### 3. Scale of the continuous covariates

Nearest neighbor uses a pooled z-score. The mean and standard deviation are taken over both arms together.

In [3]:
from core.matching import covariate_scale

covariate_scale(focal, comparison, spec["numeric"])


,covariate,pooled_mean,pooled_sd
0,age,31.338612,7.829430
1,plays_labeled_14d,11.224727,5.577466
2,plays_labeled_60d,29.866212,14.171778
3,plays_unlabeled_14d,36.418775,17.525698
4,plays_unlabeled_60d,96.942967,45.990308
5,watch_time_labeled_14d,24.277218,11.627941
6,watch_time_labeled_60d,66.156240,30.839280
7,watch_time_unlabeled_14d,66.031871,30.422810
8,watch_time_unlabeled_60d,179.432978,83.257943


### 4. Nearest neighbor inside the cell

Each focal unit takes the comparison unit in the same cell with the smallest Euclidean distance. A caliper, when there is one, sets the distance to infinity and that neighbor is dropped. The ledger counts, per cell, how many focal units were kept and how many were dropped.

In [4]:
from core.matching import match_process

pairs, ledger = match_process(
    focal,
    comparison,
    spec["exact"],
    spec["numeric"],
    spec["outcomes"],
    spec.get("caliper") or {},
)
ledger.sort_values(["n_dropped", "n_focal"], ascending=[False, False])


,city,gender,n_focal,n_comparison,n_matched,n_dropped,median_distance
3,north,male,228,211,228,0,1.637011
0,east,female,206,217,206,0,1.696527
1,east,male,196,173,196,0,1.706512
5,south,male,194,199,194,0,1.777843
2,north,female,191,209,191,0,1.769871
4,south,female,185,191,185,0,1.735149


### 5. What was dropped, and how often a comparison unit is reused

Replacement means one comparison unit may be the neighbor for more than one focal unit.

In [5]:
import pandas as pd

reuse = pairs["comparison_row"].value_counts()
pd.DataFrame([{
    "focal_units": len(focal),
    "comparison_units": len(comparison),
    "exact_cells": len(ledger),
    "dropped_no_comparison_in_cell": int(ledger.loc[ledger["n_comparison"] == 0, "n_focal"].sum()),
    "dropped_by_caliper_or_empty_neighborhood": int(ledger.loc[ledger["n_comparison"] > 0, "n_dropped"].sum()),
    "pairs": len(pairs),
    "comparison_units_used": int(reuse.shape[0]),
    "max_reuse": int(reuse.max()) if len(reuse) else 0,
}])


,focal_units,comparison_units,exact_cells,dropped_no_comparison_in_cell,dropped_by_caliper_or_empty_neighborhood,pairs,comparison_units_used,max_reuse
0,1200,1200,6,0,0,1200,697,6


### 6. Eight pairs

Exact covariates are copied onto both sides. They agree by construction. `distance` is the z-score Euclidean distance of the chosen neighbor.

In [6]:
show = ["distance"]
for column in spec["exact"]:
    show += [f"{column}_focal", f"{column}_comparison"]
for column in spec.get("caliper") or {}:
    gap = f"{column}_gap"
    pairs[gap] = (pairs[f"{column}_focal"] - pairs[f"{column}_comparison"]).abs()
    show.append(gap)
pairs[show].head(8)


,distance,city_focal,city_comparison,gender_focal,gender_comparison
0,1.509949,east,east,female,female
1,4.021472,east,east,female,female
2,1.368324,east,east,female,female
3,1.903794,east,east,female,female
4,1.428510,east,east,female,female
5,1.439087,east,east,female,female
6,1.642946,east,east,female,female
7,1.343262,east,east,female,female


### 7. Balance before and after

Standardized mean difference of each covariate, on the raw arms and on the matched pairs.

In [7]:
from core.balance import balance_before_after

balance = balance_before_after(focal, comparison, pairs, list(spec["exact"]) + list(spec["numeric"]))
balance


,covariate,mean_focal_before,mean_comparison_before,standardized_difference_before,mean_focal_after,mean_comparison_after,standardized_difference_after
0,age,31.192374,31.484850,-0.037347,31.192374,31.365771,-0.022932
1,plays_labeled_14d,11.254321,11.195132,0.010608,11.254321,10.916851,0.062902
2,plays_labeled_60d,30.184088,29.548336,0.044853,30.184088,28.459075,0.130358
3,plays_unlabeled_14d,36.621671,36.215880,0.023146,36.621671,35.026880,0.096782
4,plays_unlabeled_60d,96.603938,97.281996,-0.014738,96.603938,92.926811,0.090159
5,watch_time_labeled_14d,24.169796,24.384640,-0.018470,24.169796,23.429791,0.068604
6,watch_time_labeled_60d,67.160992,65.151488,0.065168,67.160992,64.792105,0.079847
7,watch_time_unlabeled_14d,65.653757,66.409985,-0.024849,65.653757,63.116578,0.092646
8,watch_time_unlabeled_60d,179.429323,179.436634,-0.000088,179.429323,173.033962,0.082368


### 8. The contrast

Mean of focal minus comparison. The standard error is Abadie-Imbens, with up to five within-arm neighbors. The Hodges-Lehmann median and the Wilcoxon p-value are the rank summary beside that mean.

In [8]:
from core.imbens import contrast_table

effects = contrast_table(
    focal,
    comparison,
    pairs,
    spec["outcomes"],
    spec["exact"],
    spec["numeric"],
    n_neighbors=int(cfg.get("imbens_neighbors", 5)),
)
out = root / "data" / "results"
out.mkdir(parents=True, exist_ok=True)
pairs.to_csv(out / "algorithm_pairs.csv", index=False)
effects.to_csv(out / "algorithm_effects.csv", index=False)
balance.to_csv(out / "algorithm_balance.csv", index=False)
effects


,outcome,n_pairs,hl_focal_minus_comparison,mean_difference,wilcoxon_statistic,wilcoxon_p,variance,standard_error,z_statistic,p_value,ci95_low,ci95_high
0,labeled_exposure_share,1200,-0.034203,-0.034532,47281.0,8.274566e-150,0.000001,0.001217,-28.373507,0.0,-0.036917,-0.032147
1,share_week_1,1200,-0.019834,-0.020192,135732.0,4.740454e-78,0.000001,0.001170,-17.255776,0.0,-0.022486,-0.017899
2,share_week_2,1200,-0.025146,-0.025215,94389.0,1.155065e-108,0.000001,0.001205,-20.929530,0.0,-0.027576,-0.022854
3,share_week_4,1200,-0.028721,-0.029027,71985.0,2.122000e-127,0.000001,0.001224,-23.710890,0.0,-0.031426,-0.026627
4,share_week_8,1200,-0.033478,-0.033882,48412.0,9.633126e-149,0.000001,0.001147,-29.527414,0.0,-0.036131,-0.031633
